In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PATH / "feature_dataset.csv")

print(df.shape)

(80000, 65)


In [12]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,M20,M21,M22,M40,M41,M42,M43,M60,M61,M62,...,SpectralFlatness,SpectralRollOff,SpectralBandwidth,MphiNL,SigmaDP,SigmaZ2,M2M4SNR,Modulation,OriginalSNR,Label
0,0.000026,0.000069,0.000026,3.157343e-09,2.930586e-09,6.460820e-09,2.930586e-09,3.500275e-13,4.633123e-13,4.327737e-13,...,0.627377,43,18.381460,1.0,3.797474,0.005159,1.105342,QPSK,2,7
1,0.000028,0.000068,0.000028,3.215990e-09,2.804059e-09,6.498267e-09,2.804059e-09,1.781424e-13,4.586850e-13,3.564625e-13,...,0.630078,48,19.898510,1.0,2.716261,0.005688,0.889030,QPSK,2,7
2,0.000004,0.000069,0.000004,2.910366e-09,8.069788e-10,7.002352e-09,8.069788e-10,1.588626e-13,3.876018e-13,1.401655e-13,...,0.624587,49,19.985046,1.0,3.927797,0.005326,0.360494,QPSK,2,7
3,0.000005,0.000070,0.000005,3.598735e-09,7.073600e-10,7.253493e-09,7.073600e-10,3.053956e-13,5.095078e-13,1.243595e-13,...,0.606380,47,19.097059,1.0,2.835475,0.006038,0.163557,QPSK,2,7
4,0.000018,0.000068,0.000018,2.224453e-09,2.107830e-09,6.237477e-09,2.107830e-09,1.804345e-13,2.696272e-13,2.822643e-13,...,0.602295,43,19.311969,1.0,2.253482,0.005549,1.231920,QPSK,2,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,0.000061,0.000061,0.000061,3.715596e-09,3.737109e-09,3.744306e-09,3.737109e-09,2.263675e-13,2.285522e-13,2.298726e-13,...,0.180461,12,14.900702,1.0,0.031055,0.007695,12.161031,AM-DSB,12,0
79996,0.000061,0.000061,0.000061,3.718230e-09,3.735902e-09,3.741811e-09,3.735902e-09,2.266576e-13,2.284535e-13,2.295378e-13,...,0.164442,9,14.413695,1.0,0.028139,0.007700,12.453772,AM-DSB,12,0
79997,0.000061,0.000061,0.000061,3.716149e-09,3.737786e-09,3.745027e-09,3.737786e-09,2.264257e-13,2.286164e-13,2.299415e-13,...,0.168544,11,14.605114,1.0,0.031212,0.007693,12.099712,AM-DSB,12,0
79998,0.000061,0.000061,0.000061,3.721765e-09,3.738295e-09,3.743822e-09,3.738295e-09,2.271575e-13,2.288395e-13,2.298549e-13,...,0.157253,8,14.177350,1.0,0.027188,0.007700,12.228563,AM-DSB,12,0


In [13]:
# Fill NaNs only for M2M4SNR
if "M2M4SNR" in df.columns:
    median_value = df["M2M4SNR"].median()
    df["M2M4SNR"] = df["M2M4SNR"].fillna(median_value)

print(df.isnull().sum().sum())
print(df.shape)

0
(80000, 65)


In [14]:
duplicate_columns = [
    "M22",
    "C20",
    "C21",
    "C80",
    "C84",
    "GammaMean",
    "GammaMax"
]

duplicate_columns = [
    col for col in duplicate_columns
    if col in df.columns
]

df.drop(columns=duplicate_columns, inplace=True)

print(df.shape)

(80000, 58)


In [15]:
constant_columns = [
    "WaveletASKCorrelation"
]

constant_columns = [
    col for col in constant_columns
    if col in df.columns
]

df.drop(columns=constant_columns, inplace=True)

print(df.shape)

(80000, 57)


In [16]:
redundant = [
    "M40",
    "M41",
    "M42",
    "M43",
    "M60",
    "M61",
    "M62",
    "M63",
    "M80",
    "M84"
]

redundant = [
    col for col in redundant
    if col in df.columns
]

df.drop(columns=redundant, inplace=True)

print(df.shape)

(80000, 47)


In [17]:
print(df.isnull().sum().sum())

print(np.isinf(df.select_dtypes(np.number)).sum().sum())

print(df.shape)

0
0
(80000, 47)


In [18]:
df.to_csv(
    DATA_PATH / "feature_dataset_clean.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [19]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH / "feature_dataset_clean.csv")

X = df.drop(columns=["Modulation","Label"])

y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(64000, 45)
(16000, 45)


In [20]:
X_train.to_csv(DATA_PATH / "X_train.csv", index=False)
X_test.to_csv(DATA_PATH / "X_test.csv", index=False)

y_train.to_frame("Label").to_csv(DATA_PATH / "y_train.csv", index=False)
y_test.to_frame("Label").to_csv(DATA_PATH / "y_test.csv", index=False)